# Tahap 1: Instalasi Tools

In [1]:
!pip install -q pytest coverage pytest-cov

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 6.2 MB/s eta 0:00:00


### Analisis & Penjelasan - Tahap 1: Instalasi Tools
Pada tahap ini, kita melakukan instalasi beberapa pustaka (library) Python yang dibutuhkan untuk pengujian kode:
* **`pytest`**: Framework pengujian (testing) standar untuk Python yang memudahkan pembuatan uji fungsionalitas sederhana maupun kompleks.
* **`coverage`**: Alat untuk mengukur seberapa banyak persentase baris kode (code coverage) dari program utama yang telah dieksekusi oleh skenario pengujian.
* **`pytest-cov`**: Plugin integrasi antara pytest dan coverage agar laporan pengujian dapat langsung menampilkan metrik cakupan pengujian.
```

# Tahap 2: Menyimpan File Program Utama (evaluasi_akademik.py)

In [2]:
%%writefile evaluasi_akademik.py

def konversi_nilai_ke_bobot(nilai_angka: float) -> tuple[str, float]:
    if not isinstance(nilai_angka, (int, float)):
        raise TypeError("Nilai harus berupa angka (int atau float).")
    if nilai_angka < 0 or nilai_angka > 100:
        raise ValueError("Nilai harus berada dalam rentang 0 hingga 100.")

    if nilai_angka >= 81:
        return ("A", 4.0)
    elif nilai_angka >= 76:
        return ("AB", 3.5)
    elif nilai_angka >= 66:
        return ("B", 3.0)
    elif nilai_angka >= 61:
        return ("BC", 2.5)
    elif nilai_angka >= 51:
        return ("C", 2.0)
    elif nilai_angka >= 41:
        return ("D", 1.0)
    else:
        return ("E", 0.0)


def hitung_ipk(daftar_matkul: list[dict]) -> float:
    if not daftar_matkul:
        raise ValueError("Daftar mata kuliah tidak boleh kosong.")

    total_sks = 0
    total_poin = 0.0

    for mk in daftar_matkul:
        sks = mk.get("sks", 0)
        bobot = mk.get("bobot", 0.0)

        if sks <= 0:
            raise ValueError(f"SKS untuk mata kuliah '{mk.get('nama')}' harus lebih dari 0.")
        if bobot < 0.0 or bobot > 4.0:
            raise ValueError(f"Bobot nilai '{mk.get('nama')}' tidak valid (harus 0.0 - 4.0).")

        total_sks += sks
        total_poin += sks * bobot

    return round(total_poin / total_sks, 2)


def cek_kelayakan_yudisium(
    ipk: float,
    total_sks: int,
    poin_tak: int,
    ada_nilai_e: bool,
    total_sks_d: int
) -> dict:
    if ipk < 0.0 or ipk > 4.0 or total_sks < 0 or poin_tak < 0 or total_sks_d < 0:
        raise ValueError("Input parameter akademik tidak boleh bernilai negatif atau di luar batas.")

    alasan_gagal = []

    if total_sks < 144:
        alasan_gagal.append("Total SKS kurang dari 144")
    if ipk < 2.50:
        alasan_gagal.append("IPK di bawah standar minimal 2.50")
    if ada_nilai_e:
        alasan_gagal.append("Masih terdapat mata kuliah dengan nilai E")
    if total_sks_d > 6:
        alasan_gagal.append("Total SKS nilai D melebihi batas maksimal 6 SKS")
    if poin_tak < 100:
        alasan_gagal.append("Poin TAK kurang dari 100")

    layak = len(alasan_gagal) == 0

    return {
        "layak_yudisium": layak,
        "status": "LAYAK YUDISIUM" if layak else "BELUM LAYAK",
        "catatan": ["Semua syarat terpenuhi"] if layak else alasan_gagal
    }


def tentukan_predikat_kelulusan(
    ipk: float,
    masa_studi_semester: int,
    pernah_mengulang: bool,
    ada_sanksi_akademik: bool = False
) -> str:
    if ipk < 2.50 or ipk > 4.00:
        raise ValueError("IPK tidak valid untuk penentuan predikat kelulusan (harus 2.50 - 4.00).")
    if masa_studi_semester <= 0 or masa_studi_semester > 14:
        raise ValueError("Masa studi semester tidak valid (harus 1 - 14 semester).")

    if ipk >= 3.51:
        if masa_studi_semester <= 8 and not pernah_mengulang and not ada_sanksi_akademik:
            return "Cum Laude (Dengan Pujian)"
        else:
            return "Sangat Memuaskan"
    elif ipk >= 3.01:
        return "Sangat Memuaskan"
    else:
        return "Memuaskan"

Writing evaluasi_akademik.py


### Analisis & Penjelasan - Tahap 2: Menyimpan File Program Utama
Cell ini menyimpan kode logika bisnis utama ke dalam file bernama `evaluasi_akademik.py` menggunakan command magic `%%writefile`. Di dalamnya terdapat empat fungsi utama:
1. **`konversi_nilai_ke_bobot(nilai_angka)`**: Mengonversi nilai angka (0-100) menjadi nilai huruf (A, AB, B, BC, C, D, E) beserta bobot IPK-nya (0.0 hingga 4.0). Dilengkapi validasi tipe data dan rentang nilai.
2. **`hitung_ipk(daftar_matkul)`**: Menghitung Indeks Prestasi Kumulatif berdasarkan daftar mata kuliah yang diambil (SKS dan bobot nilainya). Memiliki validasi untuk mencegah pembagian dengan nol atau SKS bernilai non-positif.
3. **`cek_kelayakan_yudisium(...)`**: Menilai kelayakan mahasiswa untuk lulus yudisium berdasarkan akumulasi SKS minimum (144), batas IPK minimal (2.50), keberadaan nilai E, batas maksimal SKS bernilai D (maksimal 6 SKS), dan minimal poin TAK (100).
4. **`tentukan_predikat_kelulusan(...)`**: Menentukan predikat kelulusan (*Cum Laude*, Sangat Memuaskan, Memuaskan) berdasarkan IPK, masa studi semester, serta status mengulang atau sanksi akademik.
```

# Tahap 3: Skenario Pengujian Awal

In [3]:
%%writefile test_evaluasi_akademik.py
import pytest
from evaluasi_akademik import (
    konversi_nilai_ke_bobot,
    hitung_ipk,
    cek_kelayakan_yudisium,
    tentukan_predikat_kelulusan
)

@pytest.mark.parametrize("nilai_input, expected_huruf, expected_bobot", [
    (85, "A", 4.0),
    (81, "A", 4.0),
    (80.9, "AB", 3.5),
    (76, "AB", 3.5),
    (70, "B", 3.0),
    (62, "BC", 2.5),
    (55, "C", 2.0),
    (45, "D", 1.0),
    (20, "E", 0.0),
])
def test_konversi_nilai_valid(nilai_input, expected_huruf, expected_bobot):
    huruf, bobot = konversi_nilai_ke_bobot(nilai_input)
    assert huruf == expected_huruf
    assert bobot == expected_bobot

def test_hitung_ipk_normal():
    data_khs = [
        {"nama": "KPPL", "sks": 3, "bobot": 4.0},
        {"nama": "Basis Data", "sks": 4, "bobot": 3.5},
        {"nama": "Jaringan Komputer", "sks": 3, "bobot": 3.0},
    ]
    assert hitung_ipk(data_khs) == 3.50

def test_yudisium_layak():
    hasil = cek_kelayakan_yudisium(
        ipk=3.45, total_sks=146, poin_tak=120, ada_nilai_e=False, total_sks_d=2
    )
    assert hasil["layak_yudisium"] is True
    assert hasil["status"] == "LAYAK YUDISIUM"

def test_predikat_cum_laude_murni():
    predikat = tentukan_predikat_kelulusan(3.75, 8, False, False)
    assert predikat == "Cum Laude (Dengan Pujian)"

def test_predikat_sangat_memuaskan_dan_memuaskan():
    assert tentukan_predikat_kelulusan(3.25, 8, False) == "Sangat Memuaskan"
    assert tentukan_predikat_kelulusan(2.80, 9, True) == "Memuaskan"

Writing test_evaluasi_akademik.py


### Analisis & Penjelasan - Tahap 3: Skenario Pengujian Awal
Di sini kita membuat file pengujian bernama `test_evaluasi_akademik.py` yang berisi beberapa skenario kasus positif (happy path):
* **`test_konversi_nilai_valid`**: Menggunakan decorator `@pytest.mark.parametrize` untuk menguji berbagai variasi nilai batas atas, batas bawah, dan nilai tengah dari setiap rentang nilai huruf.
* **`test_hitung_ipk_normal`**: Memastikan rumus kalkulasi rata-rata tertimbang IPK berjalan dengan benar dengan contoh data 3 mata kuliah.
* **`test_yudisium_layak`**: Memastikan fungsi kelayakan yudisium mengembalikan status `LAYAK YUDISIUM` apabila semua persyaratan terpenuhi.
* **`test_predikat_cum_laude_murni`** dan **`test_predikat_sangat_memuaskan_dan_memuaskan`**: Memverifikasi bahwa predikat kelulusan yang dihasilkan sesuai dengan kriteria IPK dan masa studi.
```

# Tahap 4: Menjalankan PyTest & Coverage Tahap Awal

In [4]:
!coverage run --branch -m pytest -v
!coverage report -m

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: cov-7.1.0, platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collected 13 items                                                             

test_evaluasi_akademik.py::test_konversi_nilai_valid[85-A-4.0] PASSED    [  7%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[81-A-4.0] PASSED    [ 15%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[80.9-AB-3.5] PASSED [ 23%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[76-AB-3.5] PASSED   [ 30%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[70-B-3.0] PASSED    [ 38%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[62-BC-2.5] PASSED   [ 46%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[55-C-2.0] PASSED    [ 53%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[45-D-1.0] PAS

### Analisis & Penjelasan - Tahap 4: Menjalankan PyTest & Coverage Tahap Awal
Pada tahap ini, kita mengeksekusi perintah pengujian awal:
* **`!coverage run --branch -m pytest -v`**: Menjalankan pengujian pytest dengan pelacakan percabangan logika (*branch coverage*) aktif.
* **`!coverage report -m`**: Menampilkan laporan rangkuman cakupan kode di terminal. Hasil uji awal menunjukkan bahwa coverage program `evaluasi_akademik.py` baru mencapai **74%**, dengan beberapa baris kode (seperti penanganan *error* / pengecualian dan kondisi *edge cases*) yang belum teruji (berstatus *Missing*).
```

# Tahap 5: Menampilkan Laporan Visual HTML

In [5]:
!coverage html -d htmlcov_awal
from IPython.display import IFrame
# Menampilkan halaman detail kode evaluasi_akademik.py
IFrame(src='htmlcov_awal/evaluasi_akademik_py.html', width=1000, height=600)

Wrote HTML report to ]8;;file:///content/htmlcov_awal/index.htmlhtmlcov_awal/index.html]8;;


### Analisis & Penjelasan - Tahap 5: Menampilkan Laporan Visual HTML
* **`!coverage html -d htmlcov_awal`**: Mengonversi hasil analisis *coverage* awal ke dalam format dokumen interaktif HTML dan menyimpannya di folder `htmlcov_awal`.
* **`IFrame`**: Digunakan untuk menampilkan visualisasi halaman HTML tersebut langsung di dalam Google Colab. Visualisasi ini mempermudah kita melihat baris kode mana saja yang berwarna merah (belum tereksekusi/teruji) dan mana yang berwarna hijau (sudah lolos uji).
```

# Tahap 6: Menambahkan Skenario Negative & Edge Cases

In [6]:
%%writefile -a test_evaluasi_akademik.py

def test_konversi_nilai_invalid():
    with pytest.raises(ValueError):
        konversi_nilai_ke_bobot(-5)
    with pytest.raises(ValueError):
        konversi_nilai_ke_bobot(105)
    with pytest.raises(TypeError):
        konversi_nilai_ke_bobot("Delapan Puluh")

def test_hitung_ipk_error():
    with pytest.raises(ValueError, match="tidak boleh kosong"):
        hitung_ipk([])
    with pytest.raises(ValueError, match="SKS untuk mata kuliah"):
        hitung_ipk([{"nama": "Algoritma", "sks": 0, "bobot": 4.0}])
    with pytest.raises(ValueError, match="Bobot nilai"):
        hitung_ipk([{"nama": "Algoritma", "sks": 3, "bobot": 4.5}])

def test_yudisium_tidak_layak_komplit():
    hasil = cek_kelayakan_yudisium(
        ipk=2.10, total_sks=130, poin_tak=80, ada_nilai_e=True, total_sks_d=8
    )
    assert hasil["layak_yudisium"] is False
    assert len(hasil["catatan"]) == 5

def test_yudisium_input_negatif():
    with pytest.raises(ValueError):
        cek_kelayakan_yudisium(ipk=-1.0, total_sks=144, poin_tak=100, ada_nilai_e=False, total_sks_d=0)

def test_predikat_cum_laude_gugur_syarat():
    assert tentukan_predikat_kelulusan(3.80, 9, False, False) == "Sangat Memuaskan"
    assert tentukan_predikat_kelulusan(3.80, 8, True, False) == "Sangat Memuaskan"
    assert tentukan_predikat_kelulusan(3.80, 8, False, True) == "Sangat Memuaskan"

def test_predikat_input_invalid():
    with pytest.raises(ValueError):
        tentukan_predikat_kelulusan(2.10, 8, False)
    with pytest.raises(ValueError):
        tentukan_predikat_kelulusan(3.50, 16, False)

Appending to test_evaluasi_akademik.py


### Analisis & Penjelasan - Tahap 6: Menambahkan Skenario Negative & Edge Cases
Untuk menutupi baris kode yang belum teruji di tahap awal (meningkatkan coverage dari 74% menuju 100%), kita menambahkan skenario baru ke dalam file pengujian menggunakan mode append `%%writefile -a`:
* **`test_konversi_nilai_invalid`**: Menguji penanganan kesalahan (exception) ketika input nilai berupa angka negatif, di atas 100, atau bertipe non-numerik.
* **`test_hitung_ipk_error`**: Menguji penanganan kesalahan ketika input list mata kuliah kosong, SKS bernilai nol, atau nilai bobot berada di luar rentang standar (0.0 - 4.0).
* **`test_yudisium_tidak_layak_komplit`**: Menguji kondisi mahasiswa yang gagal memenuhi seluruh 5 syarat yudisium sekaligus, memastikan semua pesan kegagalan tercatat dengan benar.
* **`test_yudisium_input_negatif`**: Menguji perlindungan fungsionalitas terhadap input parameter akademik bernilai negatif.
* **`test_predikat_cum_laude_gugur_syarat`**: Memastikan status *Cum Laude* gugur dan beralih ke *Sangat Memuaskan* apabila masa studi > 8 semester, pernah mengulang mata kuliah, atau memiliki sanksi akademik.
* **`test_predikat_input_invalid`**: Menguji input IPK di bawah batas minimum kelulusan atau masa studi di luar batas wajar (1-14 semester).
```

# Tahap 7: Eksekusi Akhir (100% Coverage) & Unduh Bukti Artefak

In [7]:
!coverage run --branch -m pytest -v
!coverage report -m
!coverage html -d htmlcov_akhir

# Mengemas kedua laporan HTML (awal & akhir) serta file .py ke dalam ZIP
!zip -r artefak_pengujian_kppl.zip evaluasi_akademik.py test_evaluasi_akademik.py htmlcov_awal htmlcov_akhir

from google.colab import files
files.download('artefak_pengujian_kppl.zip')

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: cov-7.1.0, platformdirs-4.12.2, anyio-4.15.1, langsmith-0.14.2, typeguard-4.6.0
collected 19 items                                                             

test_evaluasi_akademik.py::test_konversi_nilai_valid[85-A-4.0] PASSED    [  5%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[81-A-4.0] PASSED    [ 10%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[80.9-AB-3.5] PASSED [ 15%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[76-AB-3.5] PASSED   [ 21%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[70-B-3.0] PASSED    [ 26%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[62-BC-2.5] PASSED   [ 31%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[55-C-2.0] PASSED    [ 36%]
test_evaluasi_akademik.py::test_konversi_nilai_valid[45-D-1.0] PAS

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Analisis & Penjelasan - Tahap 7: Eksekusi Akhir & Unduh Bukti Artefak
Langkah penutup dari siklus pengujian:
1. **Eksekusi Pengujian Ulang**: Menjalankan kembali pytest beserta coverage cabang (*branch coverage*) setelah penambahan skenario negatif. Hasil pengujian kini mencapai tingkat kesempurnaan cakupan kode **100%** (semua baris logika telah tereksekusi).
2. **Pembuatan Laporan HTML Akhir**: Menyimpan laporan hasil uji cakupan 100% ke dalam folder `htmlcov_akhir`.
3. **Pengemasan ZIP (`zip -r`)**: Mengompresi seluruh berkas penting meliputi kode program (`evaluasi_akademik.py`), skenario uji lengkap (`test_evaluasi_akademik.py`), serta laporan HTML awal dan akhir ke dalam file `artefak_pengujian_kppl.zip`.
4. **Pengunduhan Otomatis (`files.download`)**: Mengunduh file zip tersebut ke komputer lokal sebagai bukti autentik keberhasilan pengujian perangkat lunak.
```